In [ ]:
# UK Stop and Search Data, 2023-02 - 2026-01
%pip install pandas
import pandas as pd
import os
import glob

# Load & Combine: (The glob and pd.concat part)
# path to main extracted folder (containing all the csv files, UK Stop and Search Data, February 2023 to January 2026)
data_path = os.path.join(os.getcwd(), "UK Stop and Search Data, February 2023 to January 2026")

# use glob to find all the csv files in the folder
all_files = glob.glob(os.path.join(data_path,"**/*.csv"), recursive=True)
print(f"Found {len(all_files)} CSV files.")

# Load and combine
# Use list comprehension to read every file, then pd.concat to stitch them together into a single dataframe
df_list = [pd.read_csv(file) for file in all_files]
stop_search_df = pd.concat(df_list, ignore_index=True)

# Success check
print(f"Total records in combined dataframe: {len(stop_search_df)}")
stop_search_df.head()

In [ ]:
%pip install sagemaker boto3 scikit-learn


In [ ]:
# --- BLOCK: DATA CLEANING ---
# These columns are either mostly empty or not useful for a linear model
cols_to_drop = [
    'Policing operation', 
    'Latitude', 
    'Longitude', 
    'Self-defined ethnicity'
 ]

# Drop them now so the rest of the processing is faster
stop_search_df = stop_search_df.drop(columns=cols_to_drop, errors='ignore')

print(f"Dataset cleaned. Columns remaining: {len(stop_search_df.columns)}")

In [ ]:
# Target Encoding: Convert "Outcome" to Outcome_Binary (0/1)

# See all unique outcomes and how many times they occur
print(stop_search_df['Outcome'].value_counts())

# Define our mapping logic
# Mark "A no further action disposal" as 0, and everything else as 1
stop_search_df['Outcome_Binary'] = stop_search_df['Outcome'].apply(
    lambda x: 0 if x == 'A no further action disposal' else 1
)

# Check the new distribution
print(stop_search_df['Outcome_Binary'].value_counts(normalize=True))

In [ ]:
# Feature Engineering: Convert the Date into month/year numbers

# Convert Date to datetime and extract features
print("Converting Date column...")
stop_search_df['Date'] = pd.to_datetime(stop_search_df['Date'], errors='coerce')
stop_search_df['Year'] = stop_search_df['Date'].dt.year
stop_search_df['Month'] = stop_search_df['Date'].dt.month
stop_search_df = stop_search_df.drop('Date', axis=1)

print("Date conversion complete!")

In [ ]:
# Categorical Encoding: Run pd.get_dummies() to turn all text into 1s and 0s. Do this to the whole DataFrame at once.

# Check data types and identify categorical columns that need encoding
print("Data types:")
print(stop_search_df.dtypes)
print("\n" + "="*50)

# Identify categorical columns (object/string types)
categorical_cols = stop_search_df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns to encode: {categorical_cols}")

# Check unique values in each categorical column to avoid memory issues
print("\nUnique value counts for categorical columns:")
for col in categorical_cols:
    unique_count = stop_search_df[col].nunique()
    print(f"{col}: {unique_count} unique values")

# Select columns for one-hot encoding (exclude very high cardinality ones)
cols_to_encode = ['Type', 'Part of a policing operation', 'Gender', 'Age range',
                  'Self-defined ethnicity', 'Officer-defined ethnicity', 'Legislation',
                  'Object of search', 'Outcome linked to object of search',
                  'Removal of more than just outer clothing']

# Keep only columns currently present to avoid KeyError on missing columns
cols_to_encode = [c for c in cols_to_encode if c in stop_search_df.columns]
missing = set(['Type', 'Part of a policing operation', 'Gender', 'Age range',
               'Self-defined ethnicity', 'Officer-defined ethnicity', 'Legislation',
               'Object of search', 'Outcome linked to object of search',
               'Removal of more than just outer clothing']) - set(cols_to_encode)
if missing:
    print(f"Skipping missing columns (already dropped or absent): {sorted(missing)}")

print(f"\nEncoding these columns with one-hot encoding: {cols_to_encode}")

# Apply one-hot encoding to selected categorical columns
print("Encoding categorical variables...")
stop_search_df_encoded = pd.get_dummies(stop_search_df, columns=cols_to_encode, drop_first=True)

print(f"Original shape: {stop_search_df.shape}")
print(f"Encoded shape: {stop_search_df_encoded.shape}")
print(f"Added {stop_search_df_encoded.shape[1] - stop_search_df.shape[1]} new columns")

# Update the dataframe
stop_search_df = stop_search_df_encoded

print("\nCategorical encoding complete!")

In [ ]:
# Reorder Columns: Ensure Outcome_Binary is the first column of the encoded DataFrame

# 1. Create list of all columns
cols = stop_search_df.columns.tolist()

# 2. Remove 'Outcome_Binary' from its current spot and put it at the front (index 0)
# Also, we should remove the original text 'Outcome' column so it doesn't confuse the model
cols.insert(0, cols.pop(cols.index('Outcome_Binary')))
if 'Outcome' in cols:
    cols.remove('Outcome')

# 3. Apply the new column order to the DataFrame
stop_search_df = stop_search_df[cols]

print(f"New column order: {stop_search_df.columns[:5].tolist()} ...")

In [ ]:
# Prepare final modeling dataframe (avoid repeating earlier cleaning/target/date steps)# Fill missing values for robust encodingstop_search_df = stop_search_df.fillna("Unknown")# Encode only columns that still existcurrent_cols = stop_search_df.columns.tolist()cols_to_encode = [    c for c in [        'Type',        'Gender',        'Age range',        'Officer-defined ethnicity',        'Legislation',        'Object of search',        'Removal of more than just outer clothing'    ] if c in current_cols]stop_search_df_encoded = pd.get_dummies(stop_search_df, columns=cols_to_encode, drop_first=True)# Build final_df with target firstif 'Outcome_Binary' in stop_search_df_encoded.columns:    cols = stop_search_df_encoded.columns.tolist()    cols.insert(0, cols.pop(cols.index('Outcome_Binary')))    final_df = stop_search_df_encoded[cols]    # Drop non-numeric leftovers if present    final_df = final_df.drop(columns=['Outcome', 'Date', 'Outcome linked to object of search'], errors='ignore')    # Force numeric dtype for training    final_df = final_df.astype('float32')    print(f"Cleanup Complete! Final DataFrame has {final_df.shape[1]} columns.")else:    print("Target column 'Outcome_Binary' not found. Check encoding step.")

In [ ]:
# Shuffle the data
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Define split points (70/20/10)
n = len(final_df)
train_df = final_df[:int(0.7*n)]
val_df = final_df[int(0.7*n):int(0.9*n)]
test_df = final_df[int(0.9*n):]

# Save as CSV with NO headers and NO index (SageMaker Requirement)
train_df.to_csv("train_final.csv", index=False, header=False)
val_df.to_csv("validation_final.csv", index=False, header=False)
test_df.to_csv("test_final.csv", index=False, header=False)

print("Files saved locally as train_final.csv, validation_final.csv, and test_final.csv")

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session

# Create SageMaker session
sess = Session()
bucket = sess.default_bucket()
prefix = "uk-stop-search-model"

print(f"Session ready. Bucket: {bucket}")

# Upload processed datasets to S3
train_path = sess.upload_data(path='train_final.csv', bucket=bucket, key_prefix=prefix)
val_path = sess.upload_data(path='validation_final.csv', bucket=bucket, key_prefix=prefix)
test_path = sess.upload_data(path='test_final.csv', bucket=bucket, key_prefix=prefix)

print("All 3 datasets uploaded to S3")
print(f"Train: {train_path}")
print(f"Validation: {val_path}")
print(f"Test: {test_path}")

In [ ]:
import os
import boto3
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session

# Resolve region + official XGBoost training image dynamically
region = boto3.Session().region_name or "us-east-2"
container = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
    image_scope="training"
)

# Resolve IAM role ARN by name (no hardcoded account ID)
role_name = os.getenv("SAGEMAKER_ROLE_NAME")
if not role_name:
    raise ValueError("Set SAGEMAKER_ROLE_NAME environment variable to your IAM role name.")
role = boto3.client("iam").get_role(RoleName=role_name)["Role"]["Arn"]

# Reconfirm session context
sess = Session()
bucket = sess.default_bucket()
prefix = "uk-stop-search-model"

print("Training environment configured")
print(f"Region: {region}")
print(f"Container: {container}")
print(f"Bucket: {bucket}")
print(f"Role: {role}")


In [ ]:
from sagemaker.core.training.configs import InputData as TrainingInput

# Create SageMaker training channel inputs
s3_input_train = TrainingInput(channel_name="train", data_source=train_path, content_type="text/csv")
s3_input_val = TrainingInput(channel_name="validation", data_source=val_path, content_type="text/csv")
s3_input_test = TrainingInput(channel_name="test", data_source=test_path, content_type="text/csv")

print("S3 training channels ready")
print("Content type set to text/csv for SageMaker XGBoost")


In [ ]:
import importlib.metadataimport pandas as pdfrom sagemaker.train.model_trainer import ModelTrainerprint(f"SageMaker package version: {importlib.metadata.version('sagemaker')}")# Compute class-imbalance weight from local training file (label is column 0)train_labels = pd.read_csv("train_final.csv", header=None, usecols=[0]).iloc[:, 0].astype(int)neg_count = int((train_labels == 0).sum())pos_count = int((train_labels == 1).sum())scale_pos_weight = round(neg_count / max(pos_count, 1), 3)print(f"Class balance in train data -> negatives: {neg_count}, positives: {pos_count}")print(f"Using scale_pos_weight={scale_pos_weight}")# Build trainer using existing setup values from earlier cellsxgb_trainer = ModelTrainer(    sagemaker_session=sess,    role=role,    training_image=container,    compute={"instance_count": 1, "instance_type": "ml.m5.large"},    output_data_config={"s3_output_path": f"s3://{bucket}/{prefix}/output"},    hyperparameters={        "objective": "binary:logistic",        "num_round": "300",        "max_depth": "6",        "eta": "0.1",        "subsample": "0.9",        "colsample_bytree": "0.8",        "min_child_weight": "5",        "gamma": "1",        "lambda": "2",        "alpha": "0.5",        "scale_pos_weight": str(scale_pos_weight),        "eval_metric": "auc"    })print("Starting SageMaker XGBoost training job...")xgb_trainer.train(input_data_config=[s3_input_train, s3_input_val], wait=True, logs=True)

In [ ]:
import boto3

region = boto3.Session().region_name or "us-east-2"
sm_client = boto3.client("sagemaker", region_name=region)

latest_jobs = sm_client.list_training_jobs(
    NameContains="sagemaker-xgboost-job",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1,
)["TrainingJobSummaries"]

if not latest_jobs:
    raise ValueError("No SageMaker XGBoost training jobs were found in this region.")

latest_training_job_name = latest_jobs[0]["TrainingJobName"]
training_job_description = sm_client.describe_training_job(TrainingJobName=latest_training_job_name)
training_job_status = training_job_description["TrainingJobStatus"]
model_artifact_s3_uri = training_job_description["ModelArtifacts"]["S3ModelArtifacts"]

print("Latest training job located")
print(f"Region: {region}")
print(f"Training job: {latest_training_job_name}")
print(f"Status: {training_job_status}")
print(f"Model artifacts: {model_artifact_s3_uri}")


In [ ]:
import os
import tarfile
from urllib.parse import urlparse

import boto3
import xgboost as xgb

parsed_artifact = urlparse(model_artifact_s3_uri)
artifact_bucket = parsed_artifact.netloc
artifact_key = parsed_artifact.path.lstrip("/")

local_tar_path = "xgb_model.tar.gz"
extract_dir = "xgb_model_artifacts"

boto3.client("s3").download_file(artifact_bucket, artifact_key, local_tar_path)
print(f"Downloaded model artifact to {local_tar_path}")

if os.path.isdir(extract_dir):
    for root, dirs, files in os.walk(extract_dir, topdown=False):
        for file_name in files:
            os.remove(os.path.join(root, file_name))
        for dir_name in dirs:
            os.rmdir(os.path.join(root, dir_name))
else:
    os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(local_tar_path, "r:gz") as tar:
    tar.extractall(path=extract_dir)

model_file_path = None
for root, _, files in os.walk(extract_dir):
    for file_name in files:
        if file_name in {"xgboost-model", "model.bin", "model"} or file_name.endswith((".model", ".bin")):
            model_file_path = os.path.join(root, file_name)
            break
    if model_file_path:
        break

if model_file_path is None:
    raise FileNotFoundError("Could not find an XGBoost model file inside the SageMaker model artifact.")

booster = xgb.Booster()
booster.load_model(model_file_path)

print(f"Loaded XGBoost model from {model_file_path}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

# Load validation and test splits written earlier for SageMaker
# Column 0 is the label; remaining columns are features.
val_eval_df = pd.read_csv("validation_final.csv", header=None)
test_eval_df = pd.read_csv("test_final.csv", header=None)

y_val = val_eval_df.iloc[:, 0].astype(int)
X_val = val_eval_df.iloc[:, 1:]
y_test = test_eval_df.iloc[:, 0].astype(int)
X_test = test_eval_df.iloc[:, 1:]

# Get raw probabilities from the trained model
y_val_prob_raw = booster.predict(xgb.DMatrix(X_val))
y_test_prob_raw = booster.predict(xgb.DMatrix(X_test))

val_auc_raw = roc_auc_score(y_val, y_val_prob_raw)
test_auc_raw = roc_auc_score(y_test, y_test_prob_raw)

# If AUC is below 0.5 on validation, invert probability direction
if val_auc_raw < 0.5:
    y_val_prob = 1.0 - y_val_prob_raw
    y_test_prob = 1.0 - y_test_prob_raw
    probability_direction = "inverted (1 - p)"
else:
    y_val_prob = y_val_prob_raw
    y_test_prob = y_test_prob_raw
    probability_direction = "raw (p)"

# Tune threshold on validation set to maximize accuracy
thresholds = np.linspace(0.05, 0.95, 181)
val_accs = [accuracy_score(y_val, (y_val_prob >= t).astype(int)) for t in thresholds]
best_threshold = float(thresholds[int(np.argmax(val_accs))])

# Apply best threshold to test set
y_pred = (y_test_prob >= best_threshold).astype(int)

# Baseline (majority class) accuracy on test set
majority_class = int(y_test.value_counts().idxmax())
baseline_pred = np.full(len(y_test), majority_class, dtype=int)
baseline_accuracy = accuracy_score(y_test, baseline_pred)

print("Test-set evaluation complete (threshold tuned on validation)")
print(f"Validation AUC raw: {val_auc_raw:.4f}")
print(f"Test AUC raw:       {test_auc_raw:.4f}")
print(f"Probability direction used: {probability_direction}")
print(f"Best threshold from validation: {best_threshold:.3f}")
print(f"Majority-class baseline accuracy (test): {baseline_accuracy:.4f}")
print(f"Model Accuracy (test): {accuracy_score(y_test, y_pred):.4f}")
print(f"Model AUC (test, calibrated): {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"Precision:             {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall:                {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1 Score:              {f1_score(y_test, y_pred, zero_division=0):.4f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
sample_index = 0
sample_features = X_test.iloc[[sample_index]]
sample_actual = int(y_test.iloc[sample_index])
sample_probability = float(booster.predict(xgb.DMatrix(sample_features))[0])
sample_prediction = int(sample_probability >= 0.5)

print(f"Sample row: {sample_index}")
print(f"Actual label: {sample_actual}")
print(f"Predicted probability of arrest/action: {sample_probability:.4f}")
print(f"Predicted label: {sample_prediction}")


In [ ]:
print("Post-training workflow ready.")
print("This notebook now validates the completed SageMaker training job, downloads the model artifact, and evaluates it locally.")
print("No endpoint is deployed in these cells, so there is no ongoing hosting cost.")
